# Build an LLM from scratch with MAX — Interactive Notebook

This notebook is a companion to the [max-llm-book](https://llm.modular.com) tutorial.
Each section maps to a chapter in the book and lets you run the GPT-2 components interactively,
inspect real tensor shapes, and generate text from pretrained weights.

**How to use this notebook:**
- Run cells top-to-bottom with *Run → Run All Cells*, or step through them one at a time.
- Each section opens with a short summary — read the corresponding book chapter for the full explanation.
- Sections 1–8 use *eager execution* (results are immediate, weights are random).
- Sections 9–12 switch to *compiled execution* with pretrained HuggingFace GPT-2 weights (~500 MB download on first run).

> **Platform note:** The setup cell auto-selects a device. On Linux/Windows with a GPU it probes
> the accelerator and uses it if the current MAX build can compile the required Mojo kernels on
> that device; otherwise it falls back to CPU. macOS (Apple Silicon) always stays on CPU because
> the Metal backend produces incorrect logits for sequence lengths > 3. The chosen device and the
> reason are printed so you can confirm.

## Setup

Import everything needed for the whole notebook, create the CPU device, and set global defaults.

In [ ]:
import inspect
import platform
import sys
from pathlib import Path

# The `gpt2_arch` package lives at the repo root. When Jupyter launches with
# this notebook's directory as cwd (the default), that root isn't on sys.path.
# Detect and prepend it so the import below works no matter how the notebook
# is launched (pixi run notebook, bare jupyter lab, nbconvert, etc).
_HERE = Path.cwd().resolve()
_REPO_ROOT = _HERE if (_HERE / "gpt2_arch").is_dir() else _HERE.parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import clear_output

import max.experimental.functional as F
from max.driver import CPU, Accelerator, Buffer, accelerator_count
from max.dtype import DType
from max.experimental.tensor import (
    Tensor,
    TensorType,
    default_device,
    default_dtype,
)
from transformers import GPT2TokenizerFast

from gpt2_arch.gpt2 import (
    GPT2Config,
    GPT2MLP,
    causal_mask,
    GPT2MultiHeadAttention,
    LayerNorm,
    GPT2Block,
    MaxGPT2Model,
    MaxGPT2LMHeadModel,
)


def _select_device():
    """Pick the best device the current MAX build can actually run on.

    - macOS (Apple Silicon): always CPU — the Metal backend produces
      incorrect logits for sequence lengths > 3.
    - Accelerator available elsewhere: probe by compiling a GELU (which
      exercises the Mojo kernel path). Fall back to CPU if the probe fails,
      e.g. on MAX nightlies that can't resolve MOGG kernel modules for the
      experimental eager API.
    - Otherwise: CPU.
    """
    if platform.system() == "Darwin":
        return CPU(), "macOS — using CPU to avoid the Metal seq_len > 3 bug."
    if accelerator_count() == 0:
        return CPU(), "No accelerator detected — using CPU."
    gpu = Accelerator()
    try:
        with default_device(gpu), default_dtype(DType.float32):
            probe = F.gelu(Tensor.ones([1, 4]), approximate="tanh")
            _ = np.from_dlpack(probe.to(CPU()))
        return gpu, "Accelerator detected — using GPU."
    except Exception as exc:  # noqa: BLE001
        return CPU(), (
            "Accelerator detected but eager GPU compilation failed "
            f"({type(exc).__name__}); falling back to CPU."
        )


DEVICE, _device_reason = _select_device()

# Set global defaults so all Tensor creation ops use the chosen device + float32.
_dev_ctx = default_device(DEVICE)
_dev_ctx.__enter__()
_dtype_ctx = default_dtype(DType.float32)
_dtype_ctx.__enter__()

# Tokenizer (downloads ~300 KB vocab files; no model weights yet).
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

plt.rcParams.update({"figure.dpi": 100, "font.size": 10})

# Brand palette used across all plots in this notebook.
BRAND_RED = "#ff552a"
BRAND_RED_LIGHT = "#ffeeea"
BRAND_PURPLE = "#637bff"
BRAND_PURPLE_LIGHT = "#f0f2fd"
BRAND_GREY = "#888888"
BRAND_GREY_LIGHT = "#f8f9fa"

BRAND_PURPLE_CMAP = LinearSegmentedColormap.from_list(
    "brand_purple", [BRAND_PURPLE_LIGHT, BRAND_PURPLE]
)
BRAND_MASK_CMAP = LinearSegmentedColormap.from_list(
    "brand_mask", [BRAND_RED, BRAND_PURPLE_LIGHT]
)

SEQ = 8  # sequence length used in shape-inspection cells

print(f"Platform : {platform.system()} / {platform.machine()}")
print(f"Python   : {sys.version.split()[0]}")
print(f"Device   : {DEVICE}  ({_device_reason})")
print(f"Tokenizer vocab size: {tokenizer.vocab_size:,}")

## Step 01 — Model configuration

📖 [Read step_01.md](../book/src/step_01.md)

`GPT2Config` holds every architectural hyperparameter. These numbers define the shape and capacity
of GPT-2: how many tokens it can represent (`vocab_size`), how far back it can look (`n_positions`),
how rich its representations are (`n_embd`), and how many layers and heads process each token.

In [ ]:
from dataclasses import asdict

config = GPT2Config()

print("GPT2Config fields:")
for k, v in asdict(config).items():
    print(f"  {k:<25} = {v}")

print()
print("Derived shapes:")
print(f"  head_dim      = n_embd / n_head = {config.n_embd} / {config.n_head} = {config.n_embd // config.n_head}")
print(f"  mlp_inner_dim = 4 x n_embd      = 4 x {config.n_embd} = {4 * config.n_embd}")

## Step 02 — Feed-forward network

📖 [Read step_02.md](../book/src/step_02.md)

`GPT2MLP` is a two-layer network: a linear expansion from 768 → 3072 dimensions,
a GELU nonlinearity, then a projection back to 768. The GELU activation is smoother
than ReLU and suppresses near-zero values rather than hard-clamping negatives to zero.

In [ ]:
mlp = GPT2MLP(4 * config.n_embd, config)

x = Tensor.ones([1, SEQ, config.n_embd])
out = mlp(x)

print(f"Input  shape: {list(x.shape)}")
print(f"Output shape: {list(out.shape)}")
print()
print("c_fc expands  768 -> 3072 (intermediate),")
print("c_proj shrinks 3072 -> 768 (back to embedding dim).")

In [ ]:
# Visualise the GELU activation function and its effect on a distribution.
t = np.linspace(-4, 4, 1000)
gelu = t * 0.5 * (1 + np.tanh(np.sqrt(2 / np.pi) * (t + 0.044715 * t**3)))

rng = np.random.default_rng(42)
pre = rng.standard_normal(3000).astype(np.float32)
post = pre * 0.5 * (1 + np.tanh(np.sqrt(2 / np.pi) * (pre + 0.044715 * pre**3)))

fig, axes = plt.subplots(1, 3, figsize=(13, 3))

axes[0].plot(t, t, color=BRAND_GREY, ls="--", alpha=0.7, label="identity")
axes[0].plot(t, gelu, color=BRAND_PURPLE, label="GELU")
axes[0].set_title("GELU activation curve")
axes[0].set_xlabel("x")
axes[0].legend()

axes[1].hist(pre, bins=50, color=BRAND_PURPLE, alpha=0.85)
axes[1].set_title("Pre-GELU activations")
axes[1].set_xlabel("Value")

axes[2].hist(post, bins=50, color=BRAND_RED, alpha=0.85)
axes[2].set_title("Post-GELU (negatives suppressed)")
axes[2].set_xlabel("Value")

plt.suptitle("GELU squashes near-zero negatives toward 0, keeping positives nearly unchanged", y=1.02)
plt.tight_layout()
plt.show()

## Step 03 — Causal masking

📖 [Read step_03.md](../book/src/step_03.md)

Autoregressive generation requires that each token only attends to earlier tokens (and itself).
The causal mask achieves this by adding −∞ to attention scores for future positions — after softmax,
those positions receive zero probability. The mask is lower-triangular.

In [ ]:
mask = causal_mask(SEQ, 0, dtype=DType.float32, device=DEVICE)
mask_np = mask.to_numpy()

print(f"Mask shape: {list(mask.shape)}  (seq_len x seq_len)")
print()
# Replace -inf with the string '-inf' for readable display
for row in mask_np:
    print("  " + "  ".join(("-inf" if np.isinf(v) else "0.0 ") for v in row))

In [ ]:
# Replace -inf with a large negative for colour mapping.
display_mask = np.where(np.isinf(mask_np), -10.0, 0.0)

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(display_mask, cmap=BRAND_MASK_CMAP, vmin=-10, vmax=0, aspect="auto")
ax.set_title("Causal attention mask (8 tokens)\nLight = attend (0), Red = blocked (−∞)")
ax.set_xlabel("Key position")
ax.set_ylabel("Query position")
ax.set_xticks(range(SEQ))
ax.set_yticks(range(SEQ))

for i in range(SEQ):
    for j in range(SEQ):
        label = "0" if j <= i else "−∞"
        color = "black" if j <= i else "white"
        ax.text(j, i, label, ha="center", va="center", fontsize=8, color=color)

plt.tight_layout()
plt.show()

## Step 04 — Multi-head attention

📖 [Read step_04.md](../book/src/step_04.md)

`GPT2MultiHeadAttention` projects hidden states to Q, K, V, splits them across 12 heads
(64 dims each), computes scaled dot-product attention with the causal mask in parallel across
all heads, then merges the head outputs back and applies the output projection.
Each head independently learns to focus on different linguistic patterns.

In [ ]:
attn = GPT2MultiHeadAttention(config)

x = Tensor.ones([1, SEQ, config.n_embd])
out = attn(x)

print(f"Input  shape: {list(x.shape)}")
print(f"Output shape: {list(out.shape)}")
print()
print(f"12 heads x {config.n_embd // config.n_head} head_dim = {config.n_embd} (embedding dim preserved)")

In [ ]:
# Extract Q and K from the combined c_attn projection, then compute attention weights in numpy.
# GPT2MultiHeadAttention.forward() returns the projected output, not the weights themselves,
# so we re-derive them here for visualisation.
x_in = Tensor.ones([1, SEQ, config.n_embd])
qkv_t = attn.c_attn(x_in)           # [1, SEQ, 3*768]
qkv_np = qkv_t.to_numpy()           # convert to numpy for slicing

head_dim = config.n_embd // config.n_head
d = config.n_embd

q_np = qkv_np[:, :, :d].reshape(1, SEQ, config.n_head, head_dim).transpose(0, 2, 1, 3)  # [1,12,8,64]
k_np = qkv_np[:, :, d:2*d].reshape(1, SEQ, config.n_head, head_dim).transpose(0, 2, 1, 3)

scores = np.matmul(q_np, k_np.transpose(0, 1, 3, 2)) / np.sqrt(head_dim)  # [1,12,8,8]
causal = np.triu(np.full((SEQ, SEQ), -1e9), k=1)
scores = scores + causal[np.newaxis, np.newaxis]
exp_s = np.exp(scores - scores.max(axis=-1, keepdims=True))
attn_weights = exp_s / exp_s.sum(axis=-1, keepdims=True)  # [1,12,8,8]
attn_weights = attn_weights[0]  # [12,8,8]

print(f"Attention weight tensor shape: {attn_weights.shape}  (heads x query_pos x key_pos)")
print(f"Row sums (should be ~1.0): {attn_weights[0].sum(axis=-1).round(3)}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 6))
for i, ax in enumerate(axes.flat):
    im = ax.imshow(attn_weights[i], cmap=BRAND_PURPLE_CMAP, vmin=0, vmax=1, aspect="auto")
    ax.set_title(f"Head {i+1}")
    ax.set_xlabel("Key pos")
    ax.set_ylabel("Query pos")
    ax.set_xticks(range(SEQ))
    ax.set_yticks(range(SEQ))

fig.suptitle(
    "Attention weights for 8 of 12 heads (random Q,K weights; causal mask applied)\n"
    "Darker = more attention. With random weights all heads look similar — "
    "pretrained weights produce diverse, specialised patterns.",
    fontsize=10,
)
plt.tight_layout()
plt.show()

## Step 05 — Layer normalization

📖 [Read step_05.md](../book/src/step_05.md)

`LayerNorm` normalises activations across the embedding dimension so that every token's
representation has zero mean and unit variance before each sub-layer. This stabilises
training and keeps activations from exploding or vanishing as they pass through 12 blocks.

In [ ]:
ln = LayerNorm(config.n_embd)

# Construct input with large mean and variance to make the normalisation visible.
rng = np.random.default_rng(1)
x_np = (rng.standard_normal((1, SEQ, config.n_embd)) * 5 + 3).astype(np.float32)
x_buf = Buffer.from_numpy(x_np).to(DEVICE)
x_t = Tensor.from_dlpack(x_buf)

out_t = ln(x_t)
out_np = out_t.to_numpy()

print("Before LayerNorm:")
print(f"  global mean = {x_np.mean():.3f},  global std = {x_np.std():.3f}")
print(f"  per-token mean range: [{x_np[0].mean(axis=-1).min():.3f}, {x_np[0].mean(axis=-1).max():.3f}]")
print()
print("After LayerNorm:")
print(f"  global mean = {out_np.mean():.3f},  global std = {out_np.std():.3f}")
print(f"  per-token mean range: [{out_np[0].mean(axis=-1).min():.6f}, {out_np[0].mean(axis=-1).max():.6f}]")

In [ ]:
per_tok_mean_in  = x_np[0].mean(axis=-1)
per_tok_std_in   = x_np[0].std(axis=-1)
per_tok_mean_out = out_np[0].mean(axis=-1)
per_tok_std_out  = out_np[0].std(axis=-1)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))

axes[0].boxplot(
    [per_tok_mean_in, per_tok_mean_out],
    labels=["Before LN", "After LN"],
    patch_artist=True,
    boxprops=dict(facecolor=BRAND_PURPLE, alpha=0.6),
)
axes[0].axhline(0, color=BRAND_RED, ls="--", lw=1, label="target=0")
axes[0].set_title("Per-token mean")
axes[0].legend()

axes[1].boxplot(
    [per_tok_std_in, per_tok_std_out],
    labels=["Before LN", "After LN"],
    patch_artist=True,
    boxprops=dict(facecolor=BRAND_RED, alpha=0.6),
)
axes[1].axhline(1, color=BRAND_GREY, ls="--", lw=1, label="target=1")
axes[1].set_title("Per-token std")
axes[1].legend()

plt.suptitle("LayerNorm drives each token's activation to mean=0, std=1")
plt.tight_layout()
plt.show()

## Step 06 — Transformer block

📖 [Read step_06.md](../book/src/step_06.md)

`GPT2Block` wires LayerNorm → Attention → residual, then LayerNorm → MLP → residual.
The residual connections (adding the block input back to the output) give gradients a
direct path through all 12 blocks during training.

> **Heads-up — random weights are *not* GPT-2's init.** MAX's default `Linear`
> initializer fills the weight matrix with std ≈ 1.0, while GPT-2's published init
> uses std = 0.02 (and scales residual projections further by 1/√(2·n_layer)). With
> the default init, each Linear amplifies its input by roughly √fan_in, so a single
> block can blow the activation magnitude up by 1000× or more. That's exactly what
> you'll see in the cell below — and the whole point of Section 9, where we replace
> these random weights with the pretrained checkpoint that actually preserves scale
> across the stack.

In [ ]:
block = GPT2Block(config)

rng = np.random.default_rng(0)
x_np = rng.standard_normal((1, SEQ, config.n_embd)).astype(np.float32)
x_buf = Buffer.from_numpy(x_np).to(DEVICE)
x_t = Tensor.from_dlpack(x_buf)

out_t = block(x_t)
out_np = out_t.to_numpy()

in_norm  = np.linalg.norm(x_np[0],   axis=-1).mean()
out_norm = np.linalg.norm(out_np[0], axis=-1).mean()

print(f"Input  avg per-token L2 norm : {in_norm:.4f}")
print(f"Output avg per-token L2 norm : {out_norm:.4f}")
print()
ratio = out_norm / in_norm
print(f"Output / input magnitude ratio : {ratio:>10.1f}x")
print()
print("With MAX's default Linear init (std ~ 1.0) the residual stream blows up by")
print("roughly sqrt(fan_in) per Linear layer. After 12 such blocks the activations")
print("would saturate the LM head (see Step 08). Pretrained weights -- loaded in")
print("Section 9 -- keep the magnitude controlled across the stack.")

## Step 07 — Stack transformer blocks

📖 [Read step_07.md](../book/src/step_07.md)

`MaxGPT2Model` stacks token embeddings (`wte`), position embeddings (`wpe`), 12 transformer
blocks, and a final layer norm. The output is a `[batch, seq_len, 768]` tensor of
contextualised hidden states — one 768-dim vector per input token.

In [ ]:
full_model = MaxGPT2Model(config)

# Use small integer token IDs (valid indices into the vocabulary).
dummy_ids = np.array([[1, 2, 3, 4, 5, 6, 7, 8]], dtype=np.int64)
id_buf = Buffer.from_numpy(dummy_ids).to(DEVICE)
id_t = Tensor.from_dlpack(id_buf)

hidden = full_model(id_t)  # [1, 8, 768]

print(f"Input  token IDs shape : {list(id_t.shape)}  (batch x seq_len)")
print(f"Output hidden states   : {list(hidden.shape)}  (batch x seq_len x n_embd)")
print()
print("Each of the 8 input tokens now has a 768-dim contextualised representation.")

In [ ]:
hidden_np = hidden.to_numpy()  # [1, 8, 768]
token_norms = np.linalg.norm(hidden_np[0], axis=-1)  # [8]

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(1, SEQ + 1), token_norms, color=BRAND_PURPLE, alpha=0.85)
ax.set_xlabel("Token position")
ax.set_ylabel("L2 norm of hidden state")
ax.set_title(
    "Per-token hidden state norms after 12 transformer blocks (random weights)\n"
    "All positions carry a similar magnitude — residual connections keep them balanced."
)
plt.tight_layout()
plt.show()

## Step 08 — Language model head

📖 [Read step_08.md](../book/src/step_08.md)

`MaxGPT2LMHeadModel` adds one final linear projection (`lm_head`) that maps each
768-dim hidden state to 50,257 logits — one per vocabulary token. Taking softmax
over those logits gives a probability distribution over the next token.

With the default random init, the activations from Section 06 are so large by the
time they reach `lm_head` that the resulting logits saturate the softmax — almost
all the probability mass lands on a single (essentially arbitrary) token, and the
entropy collapses toward 0 bits. A *trained* GPT-2 produces a sharp but informative
distribution — Section 9 swaps the random weights for the pretrained checkpoint and
the next-token distribution becomes meaningful for the first time.

In [ ]:
lm_model = MaxGPT2LMHeadModel(config)

dummy_ids = np.array([[1, 2, 3, 4, 5, 6, 7, 8]], dtype=np.int64)
id_buf = Buffer.from_numpy(dummy_ids).to(DEVICE)
id_t = Tensor.from_dlpack(id_buf)

logits_t = lm_model(id_t)          # [1, 8, 50257]
logits_np = logits_t.to_numpy()
last_logits = logits_np[0, -1, :]  # last-token logits: [50257]

print(f"Logits shape  : {list(logits_t.shape)}  (batch x seq_len x vocab_size)")
print(f"Last token    : {list(logits_t.shape[-1:])}  logits for the next token prediction")

# Softmax
exp_l = np.exp(last_logits - last_logits.max())
probs = exp_l / exp_l.sum()

entropy = -np.sum(probs * np.log(probs + 1e-12))
print(f"\nDistribution entropy  : {entropy:.2f} bits")
print(f"Max-entropy baseline  : {np.log(config.vocab_size):.2f} bits  (uniform over {config.vocab_size:,} tokens)")
if entropy < 1.0:
    print("=> Random init has saturated the logits onto essentially one token.")
elif entropy > 0.9 * np.log(config.vocab_size):
    print("=> Random weights produce a near-uniform distribution.")
else:
    print("=> Distribution sits between uniform and a delta.")

In [ ]:
top10_ids = np.argsort(probs)[-10:][::-1]
top10_probs = probs[top10_ids]
top10_tokens = [repr(tokenizer.decode([tid])) for tid in top10_ids]

fig, ax = plt.subplots(figsize=(9, 3))
bars = ax.barh(range(10), top10_probs[::-1], color=BRAND_PURPLE, alpha=0.85)
ax.set_yticks(range(10))
ax.set_yticklabels(top10_tokens[::-1])
ax.set_xlabel("Probability")
ax.set_title(
    "Top-10 next-token probabilities (random weights)\n"
    "Saturated softmax: one token captures almost all the mass."
)
plt.tight_layout()
plt.show()

## Step 09 — Loading pretrained weights

📖 [Read step_09.md](../book/src/step_09.md)

⚠️ **This cell downloads ~500 MB of pretrained GPT-2 weights from HuggingFace on first run.**
Subsequent runs use the local cache at `~/.cache/huggingface/`.

Two adaptations are needed to map HuggingFace weights into MAX:
1. **Conv1D transpose** — GPT-2 uses `Conv1D` (shape `[in, out]`) but MAX's `Linear` expects `[out, in]`.
2. **Tied embedding** — `lm_head.weight` is tied to `wte.weight` in GPT-2 small, so it isn't
   stored separately in the checkpoint; we add it explicitly.

The adapter also normalises the `transformer.` prefix on each key so the same code works whether the
checkpoint was loaded with an older transformers release (no prefix) or a recent one (prefix already present).

After this section, all later sections use the compiled pretrained model.

---

**Execution mode shift:** Sections 1–8 used *eager* execution (each `Module.forward()` ran
immediately with randomly initialised weights). From here we switch to *compiled* execution:
we create the module inside an `F.lazy()` context (weights are symbolic placeholders), then
call `.compile(token_type, weights=adapted_weights)` to build and cache the compiled graph.
The compiled model is then called like a function for inference.

In [ ]:
import torch
from transformers import GPT2LMHeadModel as HFGPT2
from max.graph.weights import WeightData

print("Downloading / loading pretrained GPT-2 weights from HuggingFace...")
hf_model = HFGPT2.from_pretrained("gpt2")
hf_state_dict = hf_model.state_dict()
print(f"Loaded {len(hf_state_dict)} tensors from HuggingFace checkpoint.")

# Show a sample of raw keys
sample_keys = list(hf_state_dict.keys())[:6]
print("\nSample HF keys:")
for k in sample_keys:
    print(f"  {k:<45} shape={tuple(hf_state_dict[k].shape)}")

In [ ]:
# Mirrors gpt2_arch/weight_adapters.py but works from PyTorch tensors.
_CONV1D = ("c_attn", "c_proj", "c_fc")
_SKIP   = (".attn.bias", ".attn.masked_bias")


def adapt_hf_weights(hf_sd: dict) -> dict:
    """Adapt HuggingFace GPT-2 weights to MAX's expected layout."""
    result = {}
    for key, tensor in hf_sd.items():
        if any(key.endswith(s) for s in _SKIP):
            continue
        # Recent transformers releases already namespace under `transformer.`;
        # older ones did not. Normalise to MAX's expected prefix in either case.
        mapped_key = key if key.startswith("transformer.") else f"transformer.{key}"
        # Force a copy so the numpy buffer is freshly allocated and 4-byte
        # aligned. Torch's `.numpy()` returns a view of torch storage, which
        # isn't always aligned to 4 bytes and trips MAX's `session.load()`.
        arr = tensor.float().numpy().copy()
        # Conv1D stores [in, out]; MAX Linear expects [out, in].
        if any(layer in mapped_key for layer in _CONV1D) and mapped_key.endswith(".weight"):
            arr = np.ascontiguousarray(arr.T)
        result[mapped_key] = WeightData.from_numpy(arr, mapped_key)

    # GPT-2 small ties lm_head.weight to wte.weight; add it explicitly.
    if "lm_head.weight" not in result:
        wte_arr = np.array(result["transformer.wte.weight"].data)
        result["lm_head.weight"] = WeightData.from_numpy(wte_arr, "lm_head.weight")

    return result


adapted = adapt_hf_weights(hf_state_dict)
print(f"Adapted {len(adapted)} weight tensors for MAX.")

# Show the Conv1D transpose for a sample key. The HF/MAX names are identical
# here (both already include the `transformer.` prefix); the only difference
# is the [in, out] -> [out, in] transpose for Conv1D weights.
sample_key = "transformer.h.0.attn.c_attn.weight"
orig_shape = tuple(hf_state_dict[sample_key].shape)
max_shape  = tuple(adapted[sample_key].data.shape)
print(f"\nConv1D transpose example ({sample_key}):")
print(f"  HF  : {orig_shape}  [in=768, out=2304]")
print(f"  MAX : {max_shape}  [out=2304, in=768]  <- transposed")

In [ ]:
# Build the model with lazy (symbolic) weights, then compile with the adapted state dict.
# This follows the same pattern as gpt2_arch/model.py _load_model().
with F.lazy(), default_device(DEVICE), default_dtype(DType.float32):
    gpt2 = MaxGPT2LMHeadModel(config)
    gpt2.to(DEVICE)

token_type = TensorType(DType.int64, ("batch", "seq_len"), device=DEVICE)
compiled_model = gpt2.compile(token_type, weights=adapted)

print("Model compiled successfully.")
print("compiled_model is now callable: compiled_model(input_tensor) -> logits tensor")

In [ ]:
# Run inference on a real prompt and compare top-10 probabilities with Section 8.
prompt = "Hello, I'm a language model"
token_ids = tokenizer.encode(prompt)

ids_np = np.array([token_ids], dtype=np.int64)
ids_buf = Buffer.from_numpy(ids_np).to(DEVICE)
ids_t = Tensor.from_dlpack(ids_buf)

logits_out = compiled_model(ids_t)                        # [1, seq_len, 50257]
logits_np = np.from_dlpack(logits_out.to(CPU()))          # move to host for numpy
last_logits = logits_np[0, -1, :]                        # last-position logits

exp_l = np.exp(last_logits - last_logits.max())
probs = exp_l / exp_l.sum()

top10_ids = np.argsort(probs)[-10:][::-1]
top10_probs = probs[top10_ids]
top10_tokens = [repr(tokenizer.decode([tid])) for tid in top10_ids]

print(f"Prompt: {repr(prompt)}")
print()
print(f"{'Token':<20} {'Prob':>8}")
print("-" * 30)
for tok, p in zip(top10_tokens, top10_probs):
    print(f"{tok:<20} {p:8.4f}")

fig, ax = plt.subplots(figsize=(9, 3))
ax.barh(range(10), top10_probs[::-1], color=BRAND_RED, alpha=0.85)
ax.set_yticks(range(10))
ax.set_yticklabels(top10_tokens[::-1])
ax.set_xlabel("Probability")
ax.set_title(
    f'Top-10 next-token probabilities after "{prompt}" (pretrained weights)\n'
    "The distribution is now sharp — pretrained weights have learned language statistics."
)
plt.tight_layout()
plt.show()

## Step 10 — KV cache configuration

📖 [Read step_10.md](../book/src/step_10.md)

`GPT2ArchConfig` exposes the attention dimensions that `max serve` needs to
pre-allocate the KV cache: `n_kv_heads`, `head_dim`, and `num_layers`.
This notebook's implementation re-processes the full token sequence on every decode
step (pedagogically clear; quadratic complexity), while a production system would
store and reuse the K and V tensors incrementally.

In [ ]:
n_kv_heads = config.n_head
head_dim   = config.n_embd // config.n_head
num_layers = config.n_layer
dtype_bytes = 2  # bfloat16 in production

# K and V tensors per layer per token: [n_kv_heads, head_dim] x 2 (K + V)
kv_per_token_bytes = n_kv_heads * head_dim * 2 * num_layers * dtype_bytes

print("KV cache dimensions derived from GPT2Config:")
print(f"  n_kv_heads  = {n_kv_heads}  (= n_head; GPT-2 uses plain MHA)")
print(f"  head_dim    = {config.n_embd} / {config.n_head} = {head_dim}")
print(f"  num_layers  = {num_layers}")
print()
print(f"KV cache memory per token (bfloat16):")
print(f"  {n_kv_heads} heads x {head_dim} head_dim x 2 (K+V) x {num_layers} layers x {dtype_bytes} bytes")
print(f"  = {kv_per_token_bytes:,} bytes = {kv_per_token_bytes / 1024:.1f} KB per token")
print()
print(f"For max sequence length ({config.n_positions} tokens):")
total_mb = kv_per_token_bytes * config.n_positions / (1024 ** 2)
print(f"  {kv_per_token_bytes:,} x {config.n_positions} = {total_mb:.2f} MB per sequence")
print()
print("max serve pre-allocates this memory so decode steps don't trigger allocations.")

## Step 11 — Pipeline model

📖 [Read step_11.md](../book/src/step_11.md)

`GPT2PipelineModel` wraps the compiled GPT-2 for `max serve`. It handles:
- `_load_model`: the `F.lazy()` + `.compile()` pattern that built `compiled_model` in Section 9
- `execute`: takes a `GPT2Inputs` buffer, runs the compiled model, returns last-token logits
- `prepare_initial_token_inputs` / `prepare_next_token_inputs`: manage the full-sequence
  replay needed since this implementation doesn't use an incremental KV cache

This section is **read-only** — the same compiled model from Section 9 is used for generation.

In [ ]:
from gpt2_arch.model import GPT2PipelineModel

print("=== GPT2PipelineModel._load_model source ===")
print(inspect.getsource(GPT2PipelineModel._load_model))
print()
print("Key pattern:")
print("  1. Enter F.lazy() context  -> Module weights become symbolic placeholders")
print("  2. Instantiate the Module  -> graph is traced, not executed")
print("  3. Call .compile()         -> JIT compiles the traced graph with real weights")
print("  4. Return compiled model   -> callable, cached, ready for production inference")

## Step 12 — Architecture registration and live generation

📖 [Read step_12.md](../book/src/step_12.md)

`arch.py` and `__init__.py` declare `GPT2PipelineModel` as a named architecture so that
`max serve --custom-architectures gpt2_arch --model-path gpt2` can discover and load it
automatically.

In the terminal, `pixi run serve` starts the OpenAI-compatible HTTP endpoint. Below we
replicate the same autoregressive generation loop using `compiled_model` from Section 9.

In [ ]:
def greedy_next_token(compiled, token_ids: list[int]) -> tuple[int, np.ndarray]:
    """Run one forward pass and return (next_token_id, full_last_probs)."""
    ids_np = np.array([token_ids], dtype=np.int64)
    ids_buf = Buffer.from_numpy(ids_np).to(DEVICE)
    ids_t = Tensor.from_dlpack(ids_buf)
    logits = compiled(ids_t)                           # [1, seq_len, 50257]
    logits_np = np.from_dlpack(logits.to(CPU()))       # move to host for numpy
    last = logits_np[0, -1, :]                         # [50257]
    exp_l = np.exp(last - last.max())
    probs = exp_l / exp_l.sum()
    return int(np.argmax(probs)), probs


GEN_PROMPT = "In the beginning"
MAX_NEW_TOKENS = 20

token_ids = list(tokenizer.encode(GEN_PROMPT))
generated_ids = []
step_probs = []   # save top-k probs at each step for the chart below

print(f"Prompt   : {repr(GEN_PROMPT)}")
print(f"Tokens   : {token_ids}")
print()
print("Generating (greedy):")

full_ids = token_ids.copy()
for step in range(MAX_NEW_TOKENS):
    next_id, probs = greedy_next_token(compiled_model, full_ids)
    generated_ids.append(next_id)
    step_probs.append(probs)
    full_ids.append(next_id)

    generated_text = tokenizer.decode(generated_ids)
    clear_output(wait=True)
    print(f"Prompt   : {repr(GEN_PROMPT)}")
    print(f"Tokens   : {token_ids}")
    print()
    print(f"Generated ({step+1}/{MAX_NEW_TOKENS} tokens): {generated_text}")

print()
print("Full sequence:")
print(f"  {GEN_PROMPT}{tokenizer.decode(generated_ids)}")

In [ ]:
# Show top-10 probabilities at 3 generation steps.
SHOW_STEPS = [0, MAX_NEW_TOKENS // 2, MAX_NEW_TOKENS - 1]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, step_idx in zip(axes, SHOW_STEPS):
    probs = step_probs[step_idx]
    context_ids = token_ids + generated_ids[:step_idx]
    context_text = tokenizer.decode(context_ids)

    top10_ids = np.argsort(probs)[-10:][::-1]
    top10_p   = probs[top10_ids]
    top10_tok = [repr(tokenizer.decode([tid])) for tid in top10_ids]

    ax.barh(range(10), top10_p[::-1], color=BRAND_PURPLE, alpha=0.85)
    ax.set_yticks(range(10))
    ax.set_yticklabels(top10_tok[::-1], fontsize=8)
    ax.set_xlabel("Probability")
    # Truncate context to last 30 chars for the title
    ctx_label = ("..." + context_text[-27:]) if len(context_text) > 30 else context_text
    ax.set_title(f'Step {step_idx+1}\nAfter: "{ctx_label}"', fontsize=9)

fig.suptitle("Top-10 next-token probabilities at three generation steps", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Full generation table: (step, token_id, token_string, top-1 probability)
print(f"{'Step':>4}  {'Token ID':>8}  {'Token string':<20}  {'Top-1 prob':>10}")
print("-" * 50)
for i, (tid, probs) in enumerate(zip(generated_ids, step_probs)):
    tok_str = repr(tokenizer.decode([tid]))
    top1_prob = probs[tid]
    print(f"{i+1:>4}  {tid:>8}  {tok_str:<20}  {top1_prob:>10.4f}")

print()
print("To serve GPT-2 via the OpenAI-compatible API, run in your terminal:")
print("  pixi run serve")